# RAG Evaluation Analysis

Interactive analysis and visualization of RAG evaluation results.

## Features
- Load and explore evaluation results
- Compare Naive, KG, and Hybrid RAG performance
- Generate visualizations
- Statistical analysis

Run `python cli.py evaluate --dataset big_tech_curated` first to generate results.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))

from src.analysis.metrics import (
    load_evaluation_results,
    calculate_summary_stats,
    compare_methods,
)
from src.analysis.visualizations import (
    plot_score_comparison,
    plot_comprehensive_comparison,
)

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Setup complete!")

## 1. Load Results

Find and load evaluation result files.

In [ ]:
# Find evaluation result files
results_dir = Path('../benchmarks/runs')
result_files = sorted(results_dir.glob('*/eval_*.json'))

print(f"Found {len(result_files)} result file(s):\n")
for i, file in enumerate(result_files, 1):
    print(f"{i}. {file.name}")

# Select latest
selected_file = result_files[-1] if result_files else None

if selected_file:
    print(f"\n✓ Selected: {selected_file.name}")
else:
    print("\n⚠ No results found. Run evaluation first.")

In [ ]:
# Load results
if selected_file:
    results = load_evaluation_results(selected_file)
    
    print("="*80)
    print("EVALUATION METADATA")
    print("="*80)
    print(f"Evaluation ID: {results['evaluation_id']}")
    print(f"Dataset: {results['dataset_name']}")
    print(f"Questions: {results['num_questions']}")
    print(f"Timestamp: {results['timestamp']}")
    print("\n✓ Results loaded!")
else:
    print("⚠ No file selected")

## 2. Summary Statistics

High-level performance comparison.

In [ ]:
# Summary statistics
if selected_file:
    comparison = compare_methods(results)
    comparison_df = pd.DataFrame(comparison).T
    display(comparison_df)

## 3. Visualizations

In [ ]:
# Score distributions
if selected_file:
    plot_score_comparison(results, 'overall_score')
    plt.show()

In [ ]:
# Comprehensive view
if selected_file:
    plot_comprehensive_comparison(results)
    plt.tight_layout()
    plt.show()

## 4. Detailed Analysis

In [ ]:
# Best performing questions (Hybrid)
if selected_file:
    method_results = results['results']['hybrid']
    df = pd.DataFrame([{
        'question': r['question'][:60] + '...',
        'score': r['overall_score']
    } for r in method_results])
    display(df.nlargest(10, 'score'))

### Challenging Questions

In [ ]:
# Lowest scoring
if selected_file:
    display(df.nsmallest(10, 'score'))

### Component Analysis

In [ ]:
# Component breakdown
if selected_file:
    components = ['retrieval_score', 'accuracy_score', 'completeness_score', 
                  'hallucination_score', 'reasoning_score']
    
    method_components = {}
    for method in ['naive', 'kg', 'hybrid']:
        method_components[method] = {
            comp: sum(r[comp] for r in results['results'][method]) / len(results['results'][method])
            for comp in components
        }
    
    component_df = pd.DataFrame(method_components).T
    display(component_df)

### Component Heatmap

In [ ]:
# Heatmap
if selected_file:
    plt.figure(figsize=(10, 6))
    sns.heatmap(component_df, annot=True, cmap='RdYlGn', vmin=0, vmax=10, 
                cbar_kws={'label': 'Score'})
    plt.title('Score Components by Method')
    plt.tight_layout()
    plt.show()